In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#11.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [ ]:
// var myBatch = ExecutionQueues[1];
// myBatch

In [ ]:
//BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

In [ ]:
OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D")

Opening existing database 'P:\hpccluster\RisingBubble3D'.


{ Session Count = 7; Grid Count = 4; Path = P:\hpccluster\RisingBubble3D }

In [ ]:
// var sessions = BoSSSshell.WorkflowMgm.Sessions;
var sessions = databases.Pick(0).Sessions;
sessions

#0: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_fixedInterfaceCheck_testcase1*	09/05/2024 15:13:05	00b7639e...
#1: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/05/2024 08:59:21	d25fc934...
#2: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:41:52	a929ee94...
#3: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	08/09/2024 11:54:11	cb26cb13...
#4: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_restart1*	08/05/2024 09:45:57	9d1f82be...
#5: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_setttingsCheck2	08/01/2024 09:27:13	eff0e8c8...
#6: RisingBubble3D	RB3D_16x16x32AMR1_k2_testcase1_setttingsCheck	04/19/2024 16:25:45	19291211...


In [ ]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
var sess = sessions.Pick(1);
sess

RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/05/2024 08:59:21	d25fc934...

In [ ]:
//sess.Delete(true);

In [ ]:
//sess.Export().WithSupersampling(2).Do()

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble3D__RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1__d25fc934-8195-4484-860f-f67ed7c942e1


C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble3D__RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1__d25fc934-8195-4484-860f-f67ed7c942e1

In [ ]:
evalSess.Add(sess);

# Evaluation - rising bubble benchmark quantities

In [ ]:
public static void PerformPostProcessing(ISessionInfo evalS) {

    // set up log 
    var db = databases.Pick(0);
    TextWriter Log = db.Controller.DBDriver.FsDriver.GetNewLog(RisingBubble2DBenchmarkQuantities.LogfileName, evalS.ID);
    string header = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "rise velocity x", "rise velocity y");
    Log.WriteLine(header);
    Log.Flush();

    // perform postprocessing for each time step
    foreach(var tStep in evalS.Timesteps) {

        // set up LsTrk and velocity fields   
        SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
        GridData grdData = (GridData)phi.GridDat; 
        LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
        levelSet.Acc(1.0, phi); 
        LevelSetTracker LsTrk = new LevelSetTracker(grdData, CutCellQuadratureMethod.Saye,  1, new string[] {"A", "B"}, levelSet);
        LsTrk.UpdateTracker(tStep.PhysicalTime);

        int D = grdData.SpatialDimension;
        XDGField[] VelocityFields = new XDGField[D];
        VelocityFields[0] = (XDGField)tStep.GetField("VelocityX");
        VelocityFields[1] = (XDGField)tStep.GetField("VelocityY");

        // compute benchmark quantities
        var R = RisingBubbleBenchmarkQuantitites.ComputeBenchmarkQuantities_RisingBubble2D(LsTrk, VelocityFields);

        // write line to log
        string line = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", tStep.TimeStepNumber.MajorNumber, tStep.PhysicalTime, 
                        R.area, R.centerX, R.centerY, R.circularity, R.MeanVelocityX, R.MeanVelocityY);
        Log.WriteLine(line);
        Log.Flush();
    }

    // Dispose
    Log.Close();
    Log.Dispose();

}

In [ ]:
List<Plot2Ddata> evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);

Element at 0: time vs area
Element at 1: time vs center of mass - x
Element at 2: time vs center of mass - y
Element at 3: time vs circularity
Element at 4: time vs rise velocity


In [ ]:
//PerformPostProcessing(evalSess.Pick(0))

In [ ]:
// try {
//     evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);
// } catch {
//     Console.WriteLine("no log file found - do postprocessing on database session");
//     PerformPostProcessing(eval)
// }

no log file found


In [ ]:
ISessionInfoExtensions.PlotData(evalData.ElementAt(3), "time", "circularity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [ ]:
ISessionInfoExtensions.PlotData(evalData.ElementAt(4), "time", "rise velocity")

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
